In [ ]:
import pandas as pd
from geopy.distance import geodesic
from datetime import datetime
import folium
import os

# Load Data
positions = pd.read_csv('../data/vessel_positions.csv')
events = pd.read_csv('../data/simulated_vessel_proximity_events.csv')

positions['datetime'] = pd.to_datetime(positions['t'], unit='ms')
events['datetime'] = pd.to_datetime(events['t'], unit='ms')

# Helper Functions

def get_latest_position(vessel_id, timestamp_ms):
    subset = positions[(positions['vessel_id'] == vessel_id) & (positions['t'] <= timestamp_ms)]
    if subset.empty:
        return None
    return subset.sort_values('t').iloc[-1]

def classify_risk(distance_nm, time_gap_sec):
    if distance_nm < 0.5 and time_gap_sec < 60:
        return "High"
    elif distance_nm < 1.0 and time_gap_sec < 120:
        return "Medium"
    elif distance_nm < 2.0 and time_gap_sec < 300:
        return "Low"
    else:
        return "Very Low"

# Modeling Risk Events

output_rows = []

for _, row in events.iterrows():
    vid1 = row['vessel_id1']
    vid2 = row['vessel_id2']
    event_time_ms = row['t']

    p1 = get_latest_position(vid1, event_time_ms)
    p2 = get_latest_position(vid2, event_time_ms)

    if p1 is not None and p2 is not None:
        pos1 = (p1['lat'], p1['lon'])
        pos2 = (p2['lat'], p2['lon'])

        distance_nm = geodesic(pos1, pos2).nautical
        time_gap_sec = abs(p1['t'] - p2['t']) / 1000

        risk = classify_risk(distance_nm, time_gap_sec)

        output_rows.append({
            'event_id': _,
            'vessel_id1': vid1,
            'vessel_id2': vid2,
            'event_time': datetime.utcfromtimestamp(event_time_ms / 1000),
            'lat': pos1[0],  # plotting using vessel 1's position
            'lon': pos1[1],
            'distance_nm': round(distance_nm, 3),
            'time_gap_sec': int(time_gap_sec),
            'risk_level': risk
        })

# Save Output Table

risk_df = pd.DataFrame(output_rows)
os.makedirs('../outputs', exist_ok=True)
risk_df.to_csv('../outputs/proximity_risk_events.csv', index=False)

# Visualize on Map

risk_map = folium.Map(location=[38.2, 24.5], zoom_start=7, tiles='CartoDB Positron')

risk_colors = {
    'High': 'red',
    'Medium': 'orange',
    'Low': 'blue',
    'Very Low': 'green'
}

for _, row in risk_df.iterrows():
    folium.CircleMarker(
        location=[row['lat'], row['lon']],
        radius=5,
        color=risk_colors.get(row['risk_level'], 'gray'),
        fill=True,
        fill_opacity=0.7,
        popup=folium.Popup(
            f"""
            <b>Risk Level:</b> {row['risk_level']}<br>
            <b>Vessels:</b> {row['vessel_id1']} & {row['vessel_id2']}<br>
            <b>Distance:</b> {row['distance_nm']} NM<br>
            <b>Time Gap:</b> {row['time_gap_sec']} sec<br>
            <b>Time:</b> {row['event_time']}
            """, max_width=250
        )
    ).add_to(risk_map)

# Save and display
risk_map.save('../outputs/proximity_risk_map.html')
risk_map


In [ ]:
from folium.plugins import HeatMap

# Heatmap of Risky Proximity Events

# Filter for meaningful risk events
heat_data = risk_df[risk_df['risk_level'].isin(['High', 'Medium'])][['lat', 'lon']].values.tolist()

# Initialize heatmap
heat_map = folium.Map(location=[38.2, 24.5], zoom_start=7, tiles='CartoDB Positron')

HeatMap(
    heat_data,
    radius=12,
    blur=15,
    min_opacity=0.3,
    gradient={0.4: 'blue', 0.65: 'orange', 1.0: 'red'}
).add_to(heat_map)

# Save and display
heat_map.save('../outputs/proximity_risk_heatmap.html')
heat_map
